## Pinecone vector database

In [ ]:
!pip install pinecone-client

In [ ]:
import pinecone

/usr/local/lib/python3.10/dist-packages/pinecone/index.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
pinecone.init(
    api_key="",
    environment="gcp-starter"
)

In [ ]:
import pinecone.info

In [ ]:
version_info = pinecone.info.version()

In [ ]:
version_info

VersionResponse(server='2.0.11', client='2.2.4')

In [ ]:
# list of indexes
pinecone.list_indexes()

['prices']

In [ ]:
# describe information about an index

pinecone.describe_index('company-profiles')

IndexDescription(name='company-profiles', metric='cosine', replicas=1, dimension=1.0, shards=1, pods=1, pod_type='starter', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')

In [ ]:
# create an index
index_name = "stock-prices"
dimension = 3
metric="cosine"

In [ ]:
# if index exist, delete it
if index_name in pinecone.list_indexes():
  pinecone.delete_index(index_name)

In [ ]:
pinecone.delete_index('stock-prices')

In [ ]:
pinecone.list_indexes()

[]

In [ ]:
pinecone.create_index(
    name=index_name,
    dimension=dimension,
    metric=metric
)

In [ ]:
pinecone.list_indexes()

['stock-prices']

In [ ]:
pinecone.describe_index('stock-prices')

IndexDescription(name='stock-prices', metric='cosine', replicas=1, dimension=3.0, shards=1, pods=1, pod_type='starter', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')

In [ ]:
# Adding pods, pod_type

pinecone.create_index(
    name=index_name,
    dimension=dimension,
    metric="cosine",
    pods=4,
    pod_type="x4"
)

In [ ]:
# access our index

index = pinecone.Index(index_name=index_name)

In [ ]:
index

### Add vector data to index

index.upsert(id,vector)
```
[(id1, vector1),(id2, vector2)]
```

In [ ]:
index.upsert(
    [
        ("TSLA",[1.,1.,1.]),
        ("IBM",[1.,2.,3.]),
        ("GM",[2.,1.,1.]),
        ("AMZN",[3.,2.,1.]),
        ("C",[2.,2.,2.])
    ]
)

{'upserted_count': 5}

In [ ]:
# see the index stats

index.describe_index_stats()

{'dimension': 3,
 'index_fullness': 5e-05,
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5}

In [ ]:
# metadata

index.upsert(
    [
        ("TSLA",[1.,1.,1.],{"stock_name":"Tesla","sector":"automobile"}),
        ("C",[2.,2.,2.], {"stock_name":"Citi","sector":"automobile"})
    ]
)

{'upserted_count': 2}

In [ ]:
# inserting data from a dataframe

data = {
    "id":["NIO","V"],
    "vector": [
        [4.,4.,4.],
        [2.,2.,5.]
    ]
}



In [ ]:
data

{'id': ['NIO', 'V'], 'vector': [[4.0, 4.0, 4.0], [2.0, 2.0, 5.0]]}

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
df

,id,vector
0,NIO,"[4.0, 4.0, 4.0]"
1,V,"[2.0, 2.0, 5.0]"


In [ ]:
index.upsert(
    vectors=(zip(df.id, df.vector))
)

{'upserted_count': 2}

In [ ]:
index.describe_index_stats()

{'dimension': 3,
 'index_fullness': 7e-05,
 'namespaces': {'': {'vector_count': 7}},
 'total_vector_count': 7}

In [ ]:
# Query the vector data

In [ ]:
index.query(
    vector=[2.,1.,2.],
    top_k = 1,
    include_values=True
)

{'matches': [{'id': 'NIO', 'score': 0.96376586, 'values': [4.0, 4.0, 4.0]}],
 'namespace': ''}

In [ ]:
index.query(
    vector=[2.,1.,2.],
    top_k = 1000,
    include_values=True,
    include_metadata=False,
    filter={
        "sector": {"$eq":"automobile"}
    }
)

{'matches': [{'id': 'TSLA', 'score': 0.96376586, 'values': [1.0, 1.0, 1.0]},
             {'id': 'C', 'score': 0.96376586, 'values': [2.0, 2.0, 2.0]}],
 'namespace': ''}

## Upserting batches

In [ ]:
pinecone.delete_index('stock-prices')

In [ ]:
pinecone.create_index('prices', dimension=64)

In [ ]:
pinecone.list_indexes()

['prices']

In [ ]:
pinecone.describe_index('prices')

IndexDescription(name='prices', metric='cosine', replicas=1, dimension=64.0, shards=1, pods=1, pod_type='starter', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')

In [ ]:
import random

ids = ['a','b','c','d']
vectors = [[random.random() for _ in range(64)] for vec in range(4)]

In [ ]:
ids

['a', 'b', 'c', 'd']

In [ ]:
len(vectors)

4

In [ ]:
index = pinecone.Index('prices')

In [ ]:
index.upsert(vectors=zip(ids, vectors))

{'upserted_count': 4}

In [ ]:
index.describe_index_stats()

{'dimension': 64,
 'index_fullness': 4e-05,
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4}

In [ ]:
# large amount of data should be inserted in batches over several requests

In [ ]:
import itertools

vector_dimension = 64
vector_count = 1000

In [ ]:
# (id, vector)

example_data_generator = map (
    lambda i :
      (f'id-{i}', [random.random() for _ in range(vector_dimension)]), range(vector_count)
)

# helper function to handle chunking of pairs

def chunks(iterable, batch_size=100):

  it = iter(iterable)
  chunk = tuple(itertools.islice(it, batch_size))

  while chunk:
    yield chunk
    chunk = tuple(itertools.islice(it, batch_size))

In [ ]:
for chunk in chunks(example_data_generator):
  index.upsert(vectors=chunk)

In [ ]:
index.describe_index_stats()

{'dimension': 64,
 'index_fullness': 0.01004,
 'namespaces': {'': {'vector_count': 1004}},
 'total_vector_count': 1004}

## Upserting batches in parallel

In [ ]:
pinecone.delete_index('prices')

In [ ]:
pinecone.create_index('prices', dimension=64)

In [ ]:
with pinecone.Index('prices', pool_threads=20) as index:

  # send request in parallel

  async_results = [
      index.upsert(vectors=ids_vectors_chunk, async_req = True)
      for ids_vectors_chunk in chunks(example_data_generator, batch_size=100)
  ]

  # wait for and retrieve responses
  [async_results.get() for async_result in async_results]

In [ ]:
index.describe_index_stats()

{'dimension': 64,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

### Upserting with Metadata

### Vector IDs must be string

In [ ]:
pinecone.list_indexes()

['test']

In [ ]:
pinecone.delete_index('companies')

In [ ]:
pinecone.create_index('test', dimension=1)

In [ ]:
ids = ['id1','id2'] # nomanclature of the IDs : uuid4, t_{timestamp}
vectors = [[0.1],[0.2]]

In [ ]:
type(ids)

list

In [ ]:
index = pinecone.Index('test')

In [ ]:
index.upsert(
    vectors = zip(ids, vectors)
)

{'upserted_count': 2}

In [ ]:
index.describe_index_stats()

{'dimension': 1,
 'index_fullness': 4e-05,
 'namespaces': {'': {'vector_count': 4}},
 'total_vector_count': 4}

### How about index dimension?

In [ ]:
pinecone.list_indexes()

['companies']

In [ ]:
pinecone.delete_index('companies')

In [ ]:
pinecone.create_index('test', dimension=2)

In [ ]:
index = pinecone.Index('test')

In [ ]:
ids = ['id1']
vectors = [[0.1]]

In [ ]:
index.upsert(vectors=zip(ids, vectors))

ApiException: ignored

### Embeddings - Sentence Transformer

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [ ]:
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 59.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 71.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 34.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.7 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=3b0cac72e0265a0babbb305759a7a789e7841842bd526ebe9ba72a932020abe2
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [ ]:
pinecone.list_indexes()

[]

In [ ]:
pinecone.delete_index('test')

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
model = SentenceTransformer('all-MiniLm-L6-v2', device='cpu')

In [ ]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
)

In [ ]:
pinecone.create_index('companies', dimension=384, metric='cosine')

In [ ]:
text_e = model.encode('This is a test')

In [ ]:
len(text_e)

384

### Manage Stock Market Data

In [ ]:
pinecone.delete_index('prices')